# Fine-tuning BERT for Named Entity Recognition (CoNLL-2003)

Professional Training mid-term project, Askary Baqir (ask084).

**Before you start:** Runtime > Change runtime type > **T4 GPU**, then run the cells in order.

Plan: fine-tune `bert-base-cased` on the CoNLL-2003 training split, choose the best epoch on the validation split, and score the test split. The model is also compared with the pre-trained `dslim/bert-base-NER` using the same word-level (first sub-token) evaluation.

In [ ]:
# 1. Check the GPU and install the extra libraries
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!pip install -q seqeval datasets
import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

CUDA available: True
GPU: Tesla T4
transformers 5.22.1 | datasets 4.8.5


In [ ]:
# 2. Load CoNLL-2003
from datasets import load_dataset

try:
    ds = load_dataset("eriktks/conll2003")
except Exception as err:
    print("Direct load failed, trying the parquet revision:", err)
    ds = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")
print(ds)

EXPECTED = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
try:
    TAGS = ds["train"].features["ner_tags"].feature.names
except Exception:
    TAGS = EXPECTED
print("Tags:", TAGS)
print("Same as the expected order:", TAGS == EXPECTED)

README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

Direct load failed, trying the parquet revision: Dataset scripts are no longer supported, but found conll2003.py


conll2003/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

conll2003/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
Tags: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Same as the expected order: True


In [ ]:
# 3. Tokenize and align labels to sub-word tokens
from transformers import AutoTokenizer, set_seed

set_seed(42)
MODEL_NAME = "bert-base-cased"      # optional second run: "distilbert-base-cased"
OUT_DIR = "ner-finetuned"           # optional second run: "ner-distil"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(batch):
    enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
    all_labels = []
    for i, tags in enumerate(batch["ner_tags"]):
        word_ids = enc.word_ids(batch_index=i)
        prev, labels = None, []
        for w in word_ids:
            if w is None:            # special tokens ([CLS], [SEP])
                labels.append(-100)
            elif w != prev:          # first sub-token of a word keeps the word's label
                labels.append(tags[w])
            else:                    # later sub-tokens are ignored by the loss
                labels.append(-100)
            prev = w
        all_labels.append(labels)
    enc["labels"] = all_labels
    return enc

tokenized = ds.map(encode, batched=True, remove_columns=ds["train"].column_names)

# Look at one example to understand the alignment (-100 = ignored)
ex = tokenized["train"][0]
print(list(zip(tok.convert_ids_to_tokens(ex["input_ids"]), ex["labels"])))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

[('[CLS]', -100), ('EU', 3), ('rejects', 0), ('German', 7), ('call', 0), ('to', 0), ('boycott', 0), ('British', 7), ('la', 0), ('##mb', -100), ('.', 0), ('[SEP]', -100)]


In [ ]:
# 4. Model, data loaders, optimizer and evaluation helper
import time, numpy as np
from torch.utils.data import DataLoader
from transformers import (AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          get_linear_schedule_with_warmup)
from seqeval.metrics import f1_score, precision_score, recall_score

device = "cuda"
id2label = {i: t for i, t in enumerate(TAGS)}
label2id = {t: i for i, t in id2label.items()}
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(TAGS), id2label=id2label, label2id=label2id).to(device)

EPOCHS, BATCH, LR, WEIGHT_DECAY, WARMUP = 3, 32, 5e-5, 0.01, 0.1
collator = DataCollatorForTokenClassification(tok)
train_dl = DataLoader(tokenized["train"], batch_size=BATCH, shuffle=True, collate_fn=collator)
val_dl = DataLoader(tokenized["validation"], batch_size=64, collate_fn=collator)
test_dl = DataLoader(tokenized["test"], batch_size=64, collate_fn=collator)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_dl)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda")

def evaluate(dl, mdl):
    """Entity-level scores using the first sub-token of each word (the usual paper protocol)."""
    mdl.eval()
    names = mdl.config.id2label          # each model has its own label order
    gold, pred = [], []
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            inputs = {k: v for k, v in batch.items() if k != "labels"}
            with torch.autocast("cuda", dtype=torch.float16):
                logits = mdl(**inputs).logits
            p = logits.argmax(-1).cpu().numpy()
            l = batch["labels"].cpu().numpy()
            for pi, li in zip(p, l):
                mask = li != -100
                gold.append([TAGS[x] for x in li[mask]])
                pred.append([names[int(x)] for x in pi[mask]])
    return f1_score(gold, pred), precision_score(gold, pred), recall_score(gold, pred)

print("Training steps:", total_steps, "| trainable parameters:", sum(p.numel() for p in model.parameters()))

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Training steps: 1317 | trainable parameters: 107726601


In [ ]:
# 5. Train (keep the best epoch by validation F1)
history, best_f1, t0 = [], 0.0, time.time()
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for batch in train_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**batch).loss
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
    f1, p, r = evaluate(val_dl, model)
    row = {"epoch": epoch + 1, "train_loss": round(running / len(train_dl), 4),
           "val_precision": round(p, 4), "val_recall": round(r, 4), "val_f1": round(f1, 4),
           "elapsed_s": round(time.time() - t0)}
    history.append(row)
    print(row)
    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained(OUT_DIR)
        tok.save_pretrained(OUT_DIR)
        print("  saved best model so far")
print("Training time (s):", round(time.time() - t0))

/tmp/ipykernel_677/2160745042.py:16: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


{'epoch': 1, 'train_loss': 0.2314, 'val_precision': np.float64(0.9293), 'val_recall': np.float64(0.9386), 'val_f1': np.float64(0.9339), 'elapsed_s': 74}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best model so far
{'epoch': 2, 'train_loss': 0.0246, 'val_precision': np.float64(0.9479), 'val_recall': np.float64(0.952), 'val_f1': np.float64(0.95), 'elapsed_s': 140}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best model so far
{'epoch': 3, 'train_loss': 0.01, 'val_precision': np.float64(0.9459), 'val_recall': np.float64(0.9537), 'val_f1': np.float64(0.9498), 'elapsed_s': 205}
Training time (s): 205


In [ ]:
# 6. Final test scores (same protocol for both models)
best = AutoModelForTokenClassification.from_pretrained(OUT_DIR).to(device)
f1, p, r = evaluate(test_dl, best)
print(f"Fine-tuned by us  | test  F1 {f1:.4f}  P {p:.4f}  R {r:.4f}")

pre = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER").to(device)
f1, p, r = evaluate(test_dl, pre)
print(f"dslim pre-trained | test  F1 {f1:.4f}  P {p:.4f}  R {r:.4f}")

print()
print("Settings:", dict(model=MODEL_NAME, epochs=EPOCHS, batch=BATCH, lr=LR, weight_decay=WEIGHT_DECAY,
                        warmup=WARMUP, seed=42, gpu=torch.cuda.get_device_name(0)))
print("History:", history)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuned by us  | test  F1 0.9139  P 0.9092  R 0.9186


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dslim pre-trained | test  F1 0.9129  P 0.9066  R 0.9193

Settings: {'model': 'bert-base-cased', 'epochs': 3, 'batch': 32, 'lr': 5e-05, 'weight_decay': 0.01, 'warmup': 0.1, 'seed': 42, 'gpu': 'Tesla T4'}
History: [{'epoch': 1, 'train_loss': 0.2314, 'val_precision': np.float64(0.9293), 'val_recall': np.float64(0.9386), 'val_f1': np.float64(0.9339), 'elapsed_s': 74}, {'epoch': 2, 'train_loss': 0.0246, 'val_precision': np.float64(0.9479), 'val_recall': np.float64(0.952), 'val_f1': np.float64(0.95), 'elapsed_s': 140}, {'epoch': 3, 'train_loss': 0.01, 'val_precision': np.float64(0.9459), 'val_recall': np.float64(0.9537), 'val_f1': np.float64(0.9498), 'elapsed_s': 205}]


In [ ]:
# 7. Zip the best model and download it
import shutil
from google.colab import files
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
files.download(OUT_DIR + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**After downloading:** unzip into `ask084/models/ner-finetuned/` on your computer. The folder must contain `config.json`, `model.safetensors` and the tokenizer files. Model weights are not committed to Git (see `.gitignore`).